In [32]:
# output_pathv = "/opt/consumer-output-delta-lake/delta-trades-batch-table"
output_path = "/opt/consumer-output-iceberg/delta-trades-batch-table"
output_path = "/opt/consumer-output-iceberg/delta-trades-batch-table"

## 1. Create spark app and read from kafka

In [33]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, DoubleType
)

# --------------------------------------------------
# 1. Create Spark session WITH DELTA LAKE SUPPORT
# --------------------------------------------------
spark = (
    SparkSession.builder
    .appName("KafkaBatchReadAndParse")
    .config("spark.ui.port", "4040")

    # ✅ BOTH extensions in ONE line (order does not matter)
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension,"
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )

    # ✅ Delta catalog (keeps Delta working)
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    ) 

    # ✅ Iceberg catalog
    .config(
        "spark.sql.catalog.iceberg",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config("spark.sql.catalog.iceberg.type", "hadoop")
    .config("spark.sql.catalog.iceberg.warehouse", "/opt/consumer-output-iceberg")

    .getOrCreate()
)

# Optional: reduce log verbosity
spark.sparkContext.setLogLevel("WARN")
# Add username here
spark.conf.set("spark.databricks.delta.commitInfo.userMetadata", "gianhubinh.le")  

# --------------------------------------------------
# 2. Read from Kafka (batch)
# --------------------------------------------------
df = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "trades.raw")
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
)

print("Number of partitions:", df.rdd.getNumPartitions())

Number of partitions: 5


## 2. Data cleaning (json value unnesting), write to delta table

### 2.1. Data cleaning

In [3]:
# --------------------------------------------------
# 3. Cast key & value from binary to string
# --------------------------------------------------
df2 = df.select(
    col("key").cast("string").alias("key"),
    col("value").cast("string").alias("value"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp")
)

# --------------------------------------------------
# 4. Define schema for JSON value
# --------------------------------------------------
value_schema = StructType([
    StructField("exchange", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("trade_id", LongType(), True),
    StructField("side", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("size", DoubleType(), True),
    StructField("exchange_time", StringType(), True),
    StructField("ingest_time", StringType(), True),
])

# --------------------------------------------------
# 5. Parse JSON string into struct
# --------------------------------------------------
df_parsed = df2.withColumn(
    "value_json",
    from_json(col("value"), value_schema)
)

# --------------------------------------------------
# 6. Unnest (flatten) JSON fields
# --------------------------------------------------
df_flat = df_parsed.select(
    col("offset"),
    col("key"),
    col("value_json.symbol").alias("symbol"),
    col("value_json.trade_id").alias("trade_id"),
    col("value_json.side").alias("side"),
    col("value_json.price").alias("price"),
    col("value_json.size").alias("size"),
    col("value_json.exchange_time").alias("exchange_time"),
    col("value_json.ingest_time").alias("ingest_time"),
)

# --------------------------------------------------
# 7. (Optional) Convert time strings to timestamp
# --------------------------------------------------
df_final = (
    df_flat
    .withColumn("exchange_time_ts", from_utc_timestamp(to_timestamp("exchange_time"), "Asia/Bangkok"))
    .withColumn("ingest_time_ts", from_utc_timestamp(to_timestamp("ingest_time"), "Asia/Bangkok"))
    .withColumn("ingest_time_minute", date_trunc("minute", col("ingest_time_ts")))
    .withColumn("exchange_second", second("exchange_time_ts"))
    .withColumn("exchange_minute", minute("exchange_time_ts"))
    .withColumn("exchange_hour", hour("exchange_time_ts"))
    .withColumn("exchange_day", dayofmonth("exchange_time_ts"))
    .withColumn("exchange_month", month("exchange_time_ts"))
    .withColumn("hour_id", date_format("ingest_time_ts", "yyyyMMddHH"))  # e.g., "2025122813"
)

# --------------------------------------------------
# 8. Show result
# --------------------------------------------------
df_final.orderBy(col("ingest_time_ts").desc()).show(5, truncate=True)

# --------------------------------------------------
# 9. Count total records
# --------------------------------------------------
# total = df_final.count()
# print("Total records:", total)

# --------------------------------------------------
# 10. WRITE TO ICEBERG FORMAT ✅
# --------------------------------------------------
# spark.sql("CREATE DATABASE IF NOT EXISTS iceberg.crypto")

# df_final.writeTo("iceberg.crypto.trades_stream") \
#     .partitionedBy(col("hour_id")) \
#     .createOrReplace()

# df_final.writeTo("iceberg.crypto.trades") \
#     .append()

print(f"Data successfully written to Delta table at: {output_path}")

+------+-------+-------+---------+----+--------+----------+--------------------+--------------------+--------------------+--------------------+-------------------+---------------+---------------+-------------+------------+--------------+----------+
|offset|    key| symbol| trade_id|side|   price|      size|       exchange_time|         ingest_time|    exchange_time_ts|      ingest_time_ts| ingest_time_minute|exchange_second|exchange_minute|exchange_hour|exchange_day|exchange_month|   hour_id|
+------+-------+-------+---------+----+--------+----------+--------------------+--------------------+--------------------+--------------------+-------------------+---------------+---------------+-------------+------------+--------------+----------+
| 22148|SOL-USD|SOL-USD|309765130|sell|  131.88|0.10907214|2026-01-03T20:50:...|2026-01-03T20:50:...|2026-01-04 03:50:...|2026-01-04 03:50:...|2026-01-04 03:50:00|             22|             50|            3|           4|             1|2026010403|
| 22

In [39]:
spark.sql("""
    CALL iceberg.system.rewrite_data_files(
        table => 'iceberg.crypto.trades_stream',
        where => 'hour_id < "2026010509"',
        options => map(
            'target-file-size-bytes', '1000',
            'min-input-files', '1'
        )
    )
""").show(truncate=False)

+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|1491                      |3                     |10548533             |0                      |
+--------------------------+----------------------+---------------------+-----------------------+



In [13]:
spark.sql("""
    SELECT 
        min(file_size_in_bytes)/1024/1024 AS min_mb,
        max(file_size_in_bytes)/1024/1024 AS max_mb,
        avg(file_size_in_bytes)/1024/1024 AS avg_mb,
        count(*) AS file_count
    FROM iceberg.crypto.trades.files
""").show()

+-------------------+------------------+----------------+----------+
|             min_mb|            max_mb|          avg_mb|file_count|
+-------------------+------------------+----------------+----------+
|0.23139286041259766|15.631911277770996|4.67881909627763|        63|
+-------------------+------------------+----------------+----------+



In [6]:
spark.sql("""
    SELECT 
        *
    FROM iceberg.crypto.trades_stream
""").show()

+------+----+-------+---------+----+--------+----------+--------------------+--------------------+--------------------+--------------+------------------+---------------+---------------+-------------+------------+--------------+----------+
|offset| key| symbol| trade_id|side|   price|      size|       exchange_time|         ingest_time|    exchange_time_ts|ingest_time_ts|ingest_time_minute|exchange_second|exchange_minute|exchange_hour|exchange_day|exchange_month|   hour_id|
+------+----+-------+---------+----+--------+----------+--------------------+--------------------+--------------------+--------------+------------------+---------------+---------------+-------------+------------+--------------+----------+
| 22912|NULL|XRP-USD|203668995| buy|  2.0044| 21.462075|2026-01-03T20:32:...|2026-01-03T20:32:...|2026-01-04 03:32:...|          NULL|              NULL|           NULL|           NULL|         NULL|        NULL|          NULL|2026010403|
| 20474|NULL|SOL-USD|309763456|sell|  131.59

In [3]:
df_parsed.select("value_json").show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------+
|value_json                                                                                                              |
+------------------------------------------------------------------------------------------------------------------------+
|{coinbase, ETH-USD, 749579196, sell, 2959.87, 0.001, 2025-12-26T14:41:50.067021Z, 2025-12-26T14:41:50.630461+00:00}     |
|{coinbase, ETH-USD, 749579197, sell, 2959.87, 0.20151392, 2025-12-26T14:41:50.067021Z, 2025-12-26T14:41:50.630601+00:00}|
|{coinbase, ETH-USD, 749579198, sell, 2959.87, 0.68501346, 2025-12-26T14:41:50.067060Z, 2025-12-26T14:41:50.630633+00:00}|
|{coinbase, ETH-USD, 749579199, sell, 2959.87, 0.74543362, 2025-12-26T14:41:50.067428Z, 2025-12-26T14:41:50.630655+00:00}|
|{coinbase, ETH-USD, 749579200, sell, 2960.04, 0.001, 2025-12-26T14:41:50.213879Z, 2025-12-26T14:41:50.781547+00:00}     |
|{coinbase, ETH-

In [4]:
df2.select("*").show(5, truncate=False)

+-------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+---------+------+-----------------------+
|key    |value                                                                                                                                                                                                                        |topic     |partition|offset|timestamp              |
+-------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+---------+------+-----------------------+
|ETH-USD|{"exchange": "coinbase", "symbol": "ETH-USD", "trade_id": 749579196, "side": "sell", "price": 2959.87, "size": 0.001, "exchange_time": "202

## 3. Joinning BTC + ETH

### 3.1. Traditional spark

In [5]:
df_eth_pri_by_day = (
    df_final
    .filter(col("symbol") == "ETH-USD")
    .groupBy("exchange_day", "exchange_hour", "exchange_minute", "exchange_second", "symbol")
    .agg(max(col("price")).alias("price_max"))
)

df_btc_pri_by_day = (
    df_final
    .filter(col("symbol") == "BTC-USD")
    .groupBy("exchange_day", "exchange_hour", "exchange_minute", "exchange_second", "symbol")
    .agg(max(col("price")).alias("price_max"))
)

df_pri_by_day = (
    df_eth_pri_by_day.alias("eth")
    .join(
        df_btc_pri_by_day.alias("btc"),
        on=["exchange_day", "exchange_hour", "exchange_minute", "exchange_second"],
        how="inner"  # change to "outer", "left", etc. if needed
    )
    .select(
        col("exchange_day"),col("exchange_hour"),col("exchange_minute"),col("exchange_second"),
        col("eth.price_max").alias("eth_price_max"),
        col("btc.price_max").alias("btc_price_max")
    )
)

df_pri_by_day.orderBy(
    col("exchange_day").asc(),
    col("exchange_hour").desc(),
    col("exchange_minute").desc(),
    col("exchange_second").desc()
).show()

df_pri_by_day.write.mode("overwrite").parquet("/opt/workspace/parquet-files")

+------------+-------------+---------------+---------------+-------------+-------------+
|exchange_day|exchange_hour|exchange_minute|exchange_second|eth_price_max|btc_price_max|
+------------+-------------+---------------+---------------+-------------+-------------+
|          26|           23|             59|             58|      2907.32|      86779.0|
|          26|           23|             59|             56|      2907.48|     86777.36|
|          26|           23|             59|             55|      2907.33|     86777.59|
|          26|           23|             59|             54|      2907.32|      86772.0|
|          26|           23|             59|             53|      2907.32|      86770.0|
|          26|           23|             59|             52|      2907.49|     86763.85|
|          26|           23|             59|             49|      2907.18|      86758.0|
|          26|           23|             59|             48|       2907.2|      86758.0|
|          26|       

### 3.2. Spark SQL alternatives (with cached)

In [6]:
# -- Register the cleaned DataFrame as a temp view (once)
df_final.createOrReplaceTempView("trades")

# -- Run the query
result = spark.sql("""
    WITH eth AS (
        SELECT 
            exchange_day,
            exchange_hour,
            exchange_minute,
            exchange_second,
            MAX(price) AS eth_price_max
        FROM trades
        WHERE symbol = 'ETH-USD'
        GROUP BY exchange_day, exchange_hour, exchange_minute, exchange_second
    ),
    btc AS (
        SELECT 
            exchange_day,
            exchange_hour,
            exchange_minute,
            exchange_second,
            MAX(price) AS btc_price_max
        FROM trades
        WHERE symbol = 'BTC-USD'
        GROUP BY exchange_day, exchange_hour, exchange_minute, exchange_second
    )
    SELECT 
        e.exchange_day,
        e.exchange_hour,
        e.exchange_minute,
        e.exchange_second,
        e.eth_price_max,
        b.btc_price_max
    FROM eth e
    INNER JOIN btc b
        ON e.exchange_day = b.exchange_day
        AND e.exchange_hour = b.exchange_hour
        AND e.exchange_minute = b.exchange_minute
        AND e.exchange_second = b.exchange_second
    ORDER BY 
        e.exchange_day ASC,
        e.exchange_hour DESC,
        e.exchange_minute DESC,
        e.exchange_second DESC
""")

result.show()

[Stage 30:==============================================>           (4 + 1) / 5]

+------------+-------------+---------------+---------------+-------------+-------------+
|exchange_day|exchange_hour|exchange_minute|exchange_second|eth_price_max|btc_price_max|
+------------+-------------+---------------+---------------+-------------+-------------+
|          26|           23|             59|             58|      2907.32|      86779.0|
|          26|           23|             59|             56|      2907.48|     86777.36|
|          26|           23|             59|             55|      2907.33|     86777.59|
|          26|           23|             59|             54|      2907.32|      86772.0|
|          26|           23|             59|             53|      2907.32|      86770.0|
|          26|           23|             59|             52|      2907.49|     86763.85|
|          26|           23|             59|             49|      2907.18|      86758.0|
|          26|           23|             59|             48|       2907.2|      86758.0|
|          26|       

In [7]:
# -- Register the cleaned DataFrame as a temp view (once)
df_final.createOrReplaceTempView("trades")

# -- Run the query
result = spark.sql("""
    SELECT 
            exchange_day,
            exchange_hour,
            exchange_minute,
            exchange_second,
            MAX(price) AS eth_price_max
        FROM trades
        WHERE symbol = 'ETH-USD'
        GROUP BY exchange_day, exchange_hour, exchange_minute, exchange_second
""")

result.show()

[Stage 35:==============================================>           (4 + 1) / 5]

+------------+-------------+---------------+---------------+-------------+
|exchange_day|exchange_hour|exchange_minute|exchange_second|eth_price_max|
+------------+-------------+---------------+---------------+-------------+
|          26|           21|             50|             58|      2921.18|
|          26|           22|              2|             10|      2896.11|
|          26|           22|              3|             54|      2898.85|
|          26|           22|              4|             22|      2913.37|
|          26|           22|              8|             47|      2908.87|
|          26|           22|              9|             38|       2910.4|
|          26|           22|             12|             10|      2903.21|
|          26|           22|             16|             47|      2904.53|
|          26|           22|             20|             38|      2914.14|
|          26|           22|             21|             44|      2914.93|
|          26|           

## 4. Transform data for candle charts

### 4.1. Chart 1: 5 mins